# Notebook 3 — Train / Validation / Test Split

## Objective

Split the labeled dataset into training, validation, and test sets
while keeping the test set isolated from the development process.

## Input Artifact

- `labeled_orders.parquet`

## Output Artifacts

- `train.parquet`
- `validation.parquet`
- `test.parquet`

In [1]:
import pandas as pd
from pathlib import Path

# Input artifact from Notebook 2
input_path = Path("../data/labeled_orders.parquet")

# Load the labeled dataset
df = pd.read_parquet(input_path)

print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumns:")
print(df.columns.tolist())

print("\nTarget column exists:", "late" in df.columns)

assert "late" in df.columns, "The 'late' target column is missing."

Dataset shape: (96476, 29)

First 5 rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,total_item_price,...,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,unique_product_count,unique_seller_count,unique_category_count,avg_latitude,avg_longitude,late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,29.99,...,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,-23.576983,-46.587161,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,118.70,...,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,-12.177924,-44.660711,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,159.90,...,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,-16.745150,-48.514783,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,45.00,...,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,-5.774190,-35.271143,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,19.90,...,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,-23.676370,-46.514627,0



Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'total_item_price', 'total_freight_value', 'avg_item_price', 'payment_count', 'total_payment_value', 'max_payment_installments', 'review_count', 'avg_review_score', 'min_review_score', 'max_review_score', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'unique_product_count', 'unique_seller_count', 'unique_category_count', 'avg_latitude', 'avg_longitude', 'late']

Target column exists: True


In [2]:
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

print(df["order_purchase_timestamp"].dtype)

datetime64[ns]


أقدم وأحدث Order

In [4]:
print("Earliest order:", df["order_purchase_timestamp"].min())
print("Latest order:", df["order_purchase_timestamp"].max())

Earliest order: 2016-09-15 12:16:38
Latest order: 2018-08-29 15:00:37


عدد Orders حسب السنة

In [5]:
orders_by_year = (
    df["order_purchase_timestamp"]
    .dt.year
    .value_counts()
    .sort_index()
)

print(orders_by_year)

order_purchase_timestamp
2016      272
2017    43426
2018    52778
Name: count, dtype: int64


توزيع الـTarget

In [6]:
print(df["late"].value_counts())

late
0    88649
1     7827
Name: count, dtype: int64


نسبة كل Class

In [7]:
print(
    df["late"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

late
0    91.89
1     8.11
Name: proportion, dtype: float64


نسبة الـLate حسب السنة

In [8]:
late_by_year = (
    df.groupby(df["order_purchase_timestamp"].dt.year)["late"]
    .mean()
    .mul(100)
    .round(2)
)

print(late_by_year)

order_purchase_timestamp
2016    1.47
2017    6.63
2018    9.37
Name: late, dtype: float64


ترتيب البيانات زمنيًا

In [9]:
df = df.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

print("Data sorted by purchase timestamp.")

Data sorted by purchase timestamp.


التأكد من الترتيب

In [10]:
print("First order:", df["order_purchase_timestamp"].iloc[0])
print("Last order:", df["order_purchase_timestamp"].iloc[-1])

First order: 2016-09-15 12:16:38
Last order: 2018-08-29 15:00:37


## Split Proportions

Use 70% of the observations for training,
15% for validation,
and 15% for testing.

In [11]:
train_ratio = 0.70
validation_ratio = 0.15
test_ratio = 0.15

print("Train ratio:", train_ratio)
print("Validation ratio:", validation_ratio)
print("Test ratio:", test_ratio)

Train ratio: 0.7
Validation ratio: 0.15
Test ratio: 0.15


## Validate Split Proportions

Verify that the train, validation, and test proportions add up to 100%.

In [12]:
print("Total ratio:", train_ratio + validation_ratio + test_ratio)

assert train_ratio + validation_ratio + test_ratio == 1.0

Total ratio: 1.0


## Calculate Split Boundaries

Calculate the row boundaries required to create the chronological
train, validation, and test sets.

In [13]:
n = len(df)

train_end = int(n * train_ratio)
validation_end = int(n * (train_ratio + validation_ratio))

print("Total rows:", n)
print("Train end:", train_end)
print("Validation end:", validation_end)

Total rows: 96476
Train end: 67533
Validation end: 82004


## Create Training Set

Create the training set using the earliest 70% of the chronologically
sorted orders.

In [14]:
train_df = df.iloc[:train_end].copy()

print("Training set shape:", train_df.shape)

Training set shape: (67533, 29)


## Create Validation Set

Create the validation set using the next 15% of the chronologically
sorted orders.

In [15]:
validation_df = df.iloc[train_end:validation_end].copy()

print("Validation set shape:", validation_df.shape)

Validation set shape: (14471, 29)


## Create Test Set

Create the test set using the most recent 15% of the chronologically
sorted orders.

In [16]:
test_df = df.iloc[validation_end:].copy()

print("Test set shape:", test_df.shape)

Test set shape: (14472, 29)


## Validate Split Sizes


In [17]:
print("Original rows:", len(df))
print("Train rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

print(
    "Total split rows:",
    len(train_df) + len(validation_df) + len(test_df)
)

assert (
    len(train_df)
    + len(validation_df)
    + len(test_df)
    == len(df)
)

Original rows: 96476
Train rows: 67533
Validation rows: 14471
Test rows: 14472
Total split rows: 96476


## Verify Temporal Boundaries

In [18]:
print("Train:")
print(train_df["order_purchase_timestamp"].min())
print(train_df["order_purchase_timestamp"].max())

print("\nValidation:")
print(validation_df["order_purchase_timestamp"].min())
print(validation_df["order_purchase_timestamp"].max())

print("\nTest:")
print(test_df["order_purchase_timestamp"].min())
print(test_df["order_purchase_timestamp"].max())

Train:
2016-09-15 12:16:38
2018-04-15 20:07:56

Validation:
2018-04-15 20:10:23
2018-06-21 07:50:39

Test:
2018-06-21 08:29:29
2018-08-29 15:00:37


## Check for Order Overlap

In [20]:
train_ids = set(train_df["order_id"])
validation_ids = set(validation_df["order_id"])
test_ids = set(test_df["order_id"])

print("Train Validation:", len(train_ids & validation_ids))
print("Train Test:", len(train_ids & test_ids))
print("Validation Test:", len(validation_ids & test_ids))

Train Validation: 0
Train Test: 0
Validation Test: 0


## Target Distribution — Training Set


In [23]:
print(train_df["late"].value_counts())

late
0    61436
1     6097
Name: count, dtype: int64


## Target Percentage — Training Set

In [24]:
print(
    train_df["late"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

late
0    90.97
1     9.03
Name: proportion, dtype: float64


## Target Distribution — Validation Set

In [25]:
print(validation_df["late"].value_counts())

late
0    13698
1      773
Name: count, dtype: int64


## Target Percentage — Validation Set

In [26]:
print(
    validation_df["late"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

late
0    94.66
1     5.34
Name: proportion, dtype: float64


## Target Distribution — Test Set

In [27]:
print(test_df["late"].value_counts()) 

late
0    13515
1      957
Name: count, dtype: int64


## Target Percentage — Test Set

In [28]:
print(
    test_df["late"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

late
0    93.39
1     6.61
Name: proportion, dtype: float64


## Validate Target Classes

In [29]:
assert set(train_df["late"].unique()) == {0, 1}
assert set(validation_df["late"].unique()) == {0, 1}
assert set(test_df["late"].unique()) == {0, 1}

print("Both target classes are present in all splits.")

Both target classes are present in all splits.


## Split Percentages


In [30]:
print("Train:", round(len(train_df) / len(df) * 100, 2), "%")
print("Validation:", round(len(validation_df) / len(df) * 100, 2), "%")
print("Test:", round(len(test_df) / len(df) * 100, 2), "%")

Train: 70.0 %
Validation: 15.0 %
Test: 15.0 %


## Save Training Artifact

In [31]:
train_path = Path("../data/train.parquet")

train_df.to_parquet(
    train_path,
    index=False
)

print("Training artifact saved to:", train_path)

Training artifact saved to: ..\data\train.parquet


## Save Validation Artifact


In [32]:
validation_path = Path("../data/validation.parquet")

validation_df.to_parquet(
    validation_path,
    index=False
)

print("Validation artifact saved to:", validation_path)

Validation artifact saved to: ..\data\validation.parquet


## Save Test Artifact


In [33]:
test_path = Path("../data/test.parquet")

test_df.to_parquet(
    test_path,
    index=False
)

print("Test artifact saved to:", test_path)

Test artifact saved to: ..\data\test.parquet


## Verify Saved Artifacts

In [34]:
print("Train exists:", train_path.exists())
print("Validation exists:", validation_path.exists())
print("Test exists:", test_path.exists())

Train exists: True
Validation exists: True
Test exists: True


## Verify Training Artifact

In [35]:
train_check = pd.read_parquet(train_path)

print("Saved training shape:", train_check.shape)

Saved training shape: (67533, 29)


## Verify Validation Artifact

In [36]:
validation_check = pd.read_parquet(validation_path)

print("Saved validation shape:", validation_check.shape)

Saved validation shape: (14471, 29)


## Verify Test Artifact

In [37]:
test_check = pd.read_parquet(test_path)

print("Saved test shape:", test_check.shape)

Saved test shape: (14472, 29)


## Final Artifact Validation

In [38]:
print("Train shape:", train_check.shape)
print("Validation shape:", validation_check.shape)
print("Test shape:", test_check.shape)

print("\nTarget column:")
print(
    "Train:", "late" in train_check.columns,
    "Validation:", "late" in validation_check.columns,
    "Test:", "late" in test_check.columns
)

print("\nChronological boundaries:")
print(
    "Train latest:", train_check["order_purchase_timestamp"].max()
)
print(
    "Validation earliest:", validation_check["order_purchase_timestamp"].min()
)
print(
    "Validation latest:", validation_check["order_purchase_timestamp"].max()
)
print(
    "Test earliest:", test_check["order_purchase_timestamp"].min()
)

assert train_check.shape == train_df.shape
assert validation_check.shape == validation_df.shape
assert test_check.shape == test_df.shape

assert "late" in train_check.columns
assert "late" in validation_check.columns
assert "late" in test_check.columns

assert (
    train_check["order_purchase_timestamp"].max()
    < validation_check["order_purchase_timestamp"].min()
)

assert (
    validation_check["order_purchase_timestamp"].max()
    < test_check["order_purchase_timestamp"].min()
)

print("\nFinal validation passed successfully.")

Train shape: (67533, 29)
Validation shape: (14471, 29)
Test shape: (14472, 29)

Target column:
Train: True Validation: True Test: True

Chronological boundaries:
Train latest: 2018-04-15 20:07:56
Validation earliest: 2018-04-15 20:10:23
Validation latest: 2018-06-21 07:50:39
Test earliest: 2018-06-21 08:29:29

Final validation passed successfully.
